# Text Embeddings: Word2Vec and BERT

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/nlp/04-embeddings/text_embeddings.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>


In [ ]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/nlp/04-embeddings"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)


Builds on frequency features from the previous lab. Dense embeddings capture word semantics in continuous vector space. Here you'll compare **static** embeddings (Word2Vec) with **contextual** embeddings (BERT) on product reviews.

**Learning objectives:** Compare Word2Vec and BERT; explore how vector size and word order affect representations.

**Installs:** `torch`, `transformers`, [Gensim](https://radimrehurek.com/gensim/intro.html#installation)


## Imports


In [3]:
%pip install -qqq torch transformers gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 58.7 MB/s eta 0:00:00


In [4]:
#imports
import numpy as np
import pandas as pd

## Gensim for word2vec
# to clean text and implement word2vec imports
from gensim.parsing.preprocessing import remove_stopwords
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec

## Pytorch imports for BERT embeddings
from transformers import AutoTokenizer, AutoModel

## Cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

# NLP Scenario
You are an analyst for a marketing company that has just launched a new product suite of mobile devices. You have data from product reviews of one of these new products, the **TechWave X1**. In this notebook your goal is to evaluate different methods for representing text with text embeddings.

#####  Product Reviews
1. "I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!"
2. "I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow."
3. "The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine."
4. "I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it."
5. "The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it."
6.  "The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."


In [5]:
# data setup
# create a list of reviews

reviews = ["I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!",
"I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow.",
"The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine.",
"I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it.",
"The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it.",
"The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."
]

# Static Embeddings

We will use the `Gensim` implementation of `Word2Vec` on our reviews to explore static embeddings. With static embeddings  a given word will have a set vector regardless of the words around it.


`Gensim` is a popular open-source library for natural language processing. It excels at topic modeling and similarity methods using models such as `Latent Dirichlet Allocation (LDA)` and `Word2Vec`. We will use `Word2Vec` to find the similarity across our reviews.

## Tokenization and pre-processing
In order to use the `Word2Vec model` we first need to convert our raw text to tokens.

We will do this by defining a function called `preprocess_text` that uses built-in preprocessing tools from `Gensim` to:
1. make all the words lowercase
2. remove special characters
3. remove very common words, known as stop words

After running this function we will create a list that contains cleaned tokens for each review. These tokens will be used in our `Word2Vec` model.

In [6]:
# Define the function
def preprocess_text(text):
    """
    Clean and tokenize text using gensim's preprocessing utilities
    """
    # Convert to lowercase and complete other basic cleaning
    tokens = simple_preprocess(text)

    # Remove stopwords
    clean_text = remove_stopwords(' '.join(tokens))
    tokens = clean_text.split()

    return tokens

In [7]:
# Preprocess the reviews by calling our preprocess_text function on each review and saving to a new list

processed_reviews = [preprocess_text(review) for review in reviews]

In [8]:
# Review our tokenized reviews

for review in processed_reviews:
    print(review)

['absolutely', 'love', 'techwave', 'daily', 'tasks', 'easier', 'efficient', 'highly', 'recommend']
['impressed', 'techwave', 'lacks', 'essential', 'features', 'slow']
['techwave', 'fantastic', 'exceeded', 'expectations', 'essential', 'daily', 'routine']
['techwave', 'average', 'job', 'particularly', 'special']
['techwave', 'terrible', 'glitches', 'crashes', 'frequently', 'regret', 'purchasing']
['techwave', 'disappointing', 'live', 'hype', 'missing', 'key', 'functionalities']


Compare the tokenized reviews with the original reviews. Note that the text has been converted to lowercase; special characters and numbers have been removed; and stopwords have also been removed.

In [23]:
reviews = pd.read_csv("../data/reviews.csv")
processed_reviews = [preprocess_text(review) for review in reviews['review']]

### Implementing `Word2Vec`

Text embeddings aim to preserve the meaning and context of words by assigning a vector to words where similar words with similar meanings or contexts would have similar vectors.

We will call [Gensims's Word2Vec](https://radimrehurek.com/gensim/models/word2vec.html) to demonstrate one example of a static embedding.
- Input:
  - Tokens for each review that have been pre-processed (cleaned)
- Output:
  - A vector (size = 100 here)


In [24]:
# Initialize the Word2Vec model on our reviews
w2v_model = Word2Vec(sentences=processed_reviews, min_count=1)

In [26]:
# Get the word vector for a word ('impressed')
w2v_model.wv['impressed']

array([ 5.1897462e-03, -9.0255598e-03,  7.1375431e-03, -1.1969609e-03,
        7.1395356e-03,  4.8736278e-03,  1.3299490e-03, -5.7085664e-03,
        5.0626243e-03,  8.9803450e-03, -2.2112348e-03,  4.3321447e-03,
       -4.8276694e-03,  5.9966012e-03,  6.5355399e-03,  1.3904953e-03,
        7.4725864e-03,  6.4667910e-03, -9.9743214e-03, -6.5421658e-03,
        9.3122404e-03, -3.4061119e-05, -2.2825405e-05, -9.4276164e-03,
       -8.4007373e-03,  1.0179044e-03,  9.8900935e-03, -3.2501246e-03,
        1.9823369e-03,  1.1911524e-03, -5.3128642e-03,  7.6724603e-03,
       -8.4231645e-03, -1.8225836e-03, -5.2788877e-03,  1.1110596e-03,
        3.3716597e-03,  9.2496239e-03, -6.2968130e-03,  1.7101779e-03,
        4.0743127e-03,  6.5115485e-03,  6.1539374e-03,  3.7661535e-03,
        1.4371434e-03, -5.7694898e-03, -5.0132570e-04,  8.2010441e-03,
        3.2744980e-03, -2.8022060e-03, -7.7011078e-03,  7.0314142e-03,
       -1.2108545e-03, -1.1739000e-03,  8.1090899e-03, -5.3126719e-03,
      

In [27]:
# and another word ('terrible')
w2v_model.wv['terrible']

array([-3.1046700e-03,  9.3682837e-03,  9.8666437e-03, -3.3700303e-04,
       -8.7955445e-03, -8.2179783e-03,  9.2855822e-03,  2.8912888e-03,
       -5.4784673e-03, -8.7270355e-03,  5.6084655e-03, -7.2344784e-03,
        2.3564277e-03, -5.4104798e-03, -7.9333354e-03, -8.4888786e-03,
        1.4308839e-03, -6.3394546e-03, -9.2988387e-03, -6.6567222e-03,
       -6.2762098e-03, -1.0842162e-04,  2.8119036e-03, -4.4679907e-03,
        5.2569844e-03,  8.8134548e-03,  9.6892362e-04, -1.5748374e-03,
       -7.5170575e-03, -3.4034121e-04, -6.1211851e-03,  4.5254533e-03,
        7.6529910e-03, -2.8316346e-03,  5.9085935e-03,  6.5985480e-03,
        8.5290652e-03, -6.6238490e-04, -6.8752901e-03, -5.7911561e-03,
       -5.7317968e-03, -4.6396484e-03,  3.3938105e-03,  4.0587327e-03,
        1.5793819e-03,  9.3673114e-03, -4.2413427e-03,  7.5323551e-05,
        9.7339936e-03,  6.9317571e-03,  8.3549209e-03, -2.8379885e-03,
        3.6135784e-03, -4.9190349e-03, -9.3527287e-03,  3.1598718e-03,
      

In [28]:
# find the most similar words to a word
w2v_model.wv.most_similar('impressed')

[('clarity', 0.28982943296432495),
 ('standby', 0.20652464032173157),
 ('palm', 0.1967487931251526),
 ('photo', 0.19108489155769348),
 ('stands', 0.18434928357601166),
 ('value', 0.18318858742713928),
 ('lighting', 0.18188413977622986),
 ('got', 0.17022094130516052),
 ('efficient', 0.16999657452106476),
 ('average', 0.16851700842380524)]

In [29]:
w2v_model.wv.most_similar('terrible')

[('clear', 0.27886027097702026),
 ('transforms', 0.27733418345451355),
 ('money', 0.27106809616088867),
 ('drops', 0.26216793060302734),
 ('natural', 0.26085472106933594),
 ('subpar', 0.245619997382164),
 ('shot', 0.23064559698104858),
 ('efficient', 0.22422440350055695),
 ('recorder', 0.2215140014886856),
 ('camera', 0.2170134037733078)]

Notice that the word 'impressed' is similar to 'essential' and 'fantastic', while the word 'terrible' is similar to 'hype' and 'disappointing'. Note some words are similar to both (such as 'easier')!

This shows that the `Word2Vec` model has learned some of the semantic relationships between words based on the context in which they appear in the reviews. Given our very small training set, these results are impressive. With more data and more examples of usage for the model to train on, these results would improve.

## Try it!
### Evaluate the impact of modifying your vector size

`Word2Vec` has multiple parameters that you can modify, and that will change the results of your model.

Below are a few key parameters you may want to edit, and their default values.

- `vector_size: int = 100`
  - [Vector size docs](https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html#vector-size)
- `window: int = 5`
  - The maximum number of words around each word that are used within any given sentence.
- `min_count: int = 5`
  - The minimum number of times a word must appear in the corpus for it to be included in the model.
  - [Min Count Docs](https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html#min-count)

#### Example with default values
```python
Word2Vec(sentences=processed_reviews, vector_size=100, window=5, min_count=5)
```

Give it a try! Modify your `Word2Vec` parameters in each exercise below and observe your output.

#### Smaller vector
- Modify the `Word2Vec` model to use a smaller vector size (e.g. 10)
- Then, retrain the model on the processed reviews and find the most similar words to the word "impressed".
- What differences do you notice in the results compared to the previous model?

In [ ]:
# Initialize the word2vec model on our reviews with a smaller vector size
w2v_model = Word2Vec(sentences=processed_reviews, vector_size=10, window=5, min_count=1)

# Find the most similar words to 'impressed'
w2v_model.wv.most_similar('impressed')

#### Larger Vector and Window
- Repeat the process and modify the `Word2Vec` model to use a larger vector size (e.g. 300) and a larger window size (e.g. 10).
- What differences do you notice in the results compared to the previous models?

In [ ]:
# Initialize the Word2Vec model on our reviews with a smaller vector size
w2v_model = Word2Vec(sentences=processed_reviews, vector_size=300, window=10, min_count=1)

# Find the most similar words to 'impressed'
w2v_model.wv.most_similar('impressed')

Notice that the `Word2Vec` model is sensitive to the vector size and other hyperparameters. These choices can affect the quality of the word embeddings and the similarity results. Results are also impacted by your training data.

Here we only have a small set of reviews, so the word embeddings may not be as accurate as they would be with a larger dataset.

## Understanding word order

Let's examine how word order affects meaning with these two sentences:

- "The supplier agreed to pay the manufacturer"
- "The manufacturer agreed to pay the supplier"

The meaning is changed depending on which words come first. Let's first make embeddings using a static embedding (`Word2Vec`), then compare the difference when we use a contextual embedding.

## Try it!
Use the steps above to convert the following two sentences using `Word2Vec`

In [30]:
# Create a list of sentences
sentences = ["The supplier agreed to pay the manufacturer",
             "The manufacturer agreed to pay the supplier"]


# Preprocess the sentences using the function we created earlier, preprocess_text()
processed_reviews = [preprocess_text(sentence) for sentence in sentences]

# Initialize the Word2Vec model
w2v_model= Word2Vec(sentences=processed_reviews, vector_size=100, window=5, min_count=1)

In [31]:
# Get the word vector for a word ('supplier')
w2v_model.wv['supplier']

array([-8.2426779e-03,  9.2993546e-03, -1.9766092e-04, -1.9672764e-03,
        4.6036304e-03, -4.0953159e-03,  2.7431143e-03,  6.9399667e-03,
        6.0654259e-03, -7.5107943e-03,  9.3823504e-03,  4.6718083e-03,
        3.9661205e-03, -6.2435055e-03,  8.4599797e-03, -2.1501649e-03,
        8.8251876e-03, -5.3620026e-03, -8.1294188e-03,  6.8245591e-03,
        1.6711927e-03, -2.1985089e-03,  9.5136007e-03,  9.4938548e-03,
       -9.7740470e-03,  2.5052286e-03,  6.1566923e-03,  3.8724565e-03,
        2.0227872e-03,  4.3050171e-04,  6.7363144e-04, -3.8206363e-03,
       -7.1402504e-03, -2.0888723e-03,  3.9238976e-03,  8.8186832e-03,
        9.2591504e-03, -5.9759365e-03, -9.4026709e-03,  9.7643770e-03,
        3.4297847e-03,  5.1661171e-03,  6.2823449e-03, -2.8042626e-03,
        7.3227035e-03,  2.8302716e-03,  2.8710044e-03, -2.3803699e-03,
       -3.1282497e-03, -2.3701417e-03,  4.2764368e-03,  7.6057913e-05,
       -9.5842788e-03, -9.6655441e-03, -6.1481940e-03, -1.2856961e-04,
      

In [32]:
# Get the word vector for a word ('manufacturer')
w2v_model.wv['manufacturer']

array([-5.3622725e-04,  2.3643136e-04,  5.1033497e-03,  9.0092728e-03,
       -9.3029495e-03, -7.1168090e-03,  6.4588725e-03,  8.9729885e-03,
       -5.0154282e-03, -3.7633716e-03,  7.3805046e-03, -1.5334714e-03,
       -4.5366134e-03,  6.5540518e-03, -4.8601604e-03, -1.8160177e-03,
        2.8765798e-03,  9.9187379e-04, -8.2852151e-03, -9.4488179e-03,
        7.3117660e-03,  5.0702621e-03,  6.7576934e-03,  7.6286553e-04,
        6.3508903e-03, -3.4053659e-03, -9.4640139e-04,  5.7685734e-03,
       -7.5216377e-03, -3.9361035e-03, -7.5115822e-03, -9.3004224e-04,
        9.5381187e-03, -7.3191668e-03, -2.3337686e-03, -1.9377411e-03,
        8.0774371e-03, -5.9308959e-03,  4.5162440e-05, -4.7537340e-03,
       -9.6035507e-03,  5.0072931e-03, -8.7595852e-03, -4.3918253e-03,
       -3.5099984e-05, -2.9618145e-04, -7.6612402e-03,  9.6147433e-03,
        4.9820580e-03,  9.2331432e-03, -8.1579173e-03,  4.4957981e-03,
       -4.1370760e-03,  8.2453608e-04,  8.4986202e-03, -4.4621765e-03,
      

In [35]:
# Stretch: calculate the cosine similarity of the same words in two sentences (with two meanings)
# Reminder: we imported cosine similarity from sklearn.metrics.pairwise at the start of this notebook

cos_sim = cosine_similarity([w2v_model.wv['supplier']], [w2v_model.wv['supplier']])
print(cos_sim)

[[0.99999994]]


#### What did you notice?

These two sentences would have identical representations in `Word2Vec` but have very different meanings. In this example, the order of the words is critical to the meaning of the phrase. We aren't able to capture this order with the bag-of-words approach used in `Word2Vec` and other similar models.

To address this issue, methods were developed to capture the sequential order of words.

# Contextual Embeddings

Unlike static embeddings, such as `Word2Vec`, **contextual embeddings** create different embeddings for the same word depending on the context in which it is used. To capture the context we use information such as the words around our word of interest, its sentence position, and other factors to map its vector.

**In other words, contextual embeddings are position aware.**

Let's look at our example from before:
- "The supplier agreed to pay the manufacturer"
- "The manufacturer agreed to pay the supplier"

We will use `BERT`, a famous language model that uses transformer encoder architecture, to create contextual embeddings.

We can access this model using the [Transformers library in HuggingFace](https://huggingface.co/docs/transformers/en/index).

In [36]:
# Example sentences
sentence1 = "The supplier agreed to pay the manufacturer"
sentence2 = "The manufacturer agreed to pay the supplier"

In [37]:
# Load BERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_model = AutoModel.from_pretrained('bert-base-uncased')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The first step is to create our tokens.  
_Note: **tensors** are a general term for algebraic objects including vectors, and matrices.  You can [learn more about tensors here](https://en.wikipedia.org/wiki/Tensor)._

In [38]:
# Add special tokens and convert to a tensor
input1 = tokenizer(sentence1, return_tensors="pt", padding=True, truncation=True)

# Take a look at the tokenized sentence
tokenizer.convert_ids_to_tokens(input1['input_ids'][0])

['[CLS]',
 'the',
 'supplier',
 'agreed',
 'to',
 'pay',
 'the',
 'manufacturer',
 '[SEP]']

Notice that the `BERT` tokenizer added two special tokens:
- `[CLS]` is BERT's way of marking the start of the input document (a sentence in this case).
- `[SEP]` indicates a separation (such as between sentences) and the end of a document.

## Try it!
Create the tokens for sentence 2.

In [39]:
# Add special tokens and convert to a tensor
input2 = tokenizer(sentence2, return_tensors="pt", padding=True, truncation=True)

# Take a look at the tokenized sentence
tokenizer.convert_ids_to_tokens(input2['input_ids'][0])

['[CLS]',
 'the',
 'manufacturer',
 'agreed',
 'to',
 'pay',
 'the',
 'supplier',
 '[SEP]']

## Setup code for `BERT` embeddings

Let's define 2 functions to create the tokens and then use them to create the `BERT` embeddings at the sentence level and the token level. For now, focus on the outputs.

How this code block works in detail is beyond the scope of this lesson.

In [40]:
# Helper function to get BERT embeddings and cosine similarity scores


def get_bert_embedding(sentence):
    """
    Get BERT embeddings for a sentence
    """
    # Tokenize and get model outputs
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True)

    # Get model outputs
    outputs = bert_model(**inputs)

    # Get embeddings from last hidden state and convert to numpy
    embeddings = outputs.last_hidden_state[0].detach().numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    return embeddings, tokens

def compare_word(word, emb1, tokens1, emb2, tokens2):
    """
    Compare BERT embeddings for a specific word in two embeddings
    """

    # Find the word indices
    idx1 = tokens1.index(word)
    idx2 = tokens2.index(word)

    # Get the word vectors
    vec1 = emb1[idx1]
    vec2 = emb2[idx2]

    # Calculate cosine similarity
    similarity = cosine_similarity(vec1.reshape(1, -1), vec2.reshape(1, -1))[0][0]

    print(f"Comparing embeddings for word '{word}':")
    print(f"Similarity score: {similarity:.4f}")
    print(f"Vector in sentence 1 (first 5 values): {vec1[:5]}")
    print(f"Vector in sentence 2 (first 5 values): {vec2[:5]}")

    return similarity

Now we can pass each of our sentences to the function to get the embeddings for each.

In [41]:
# Print each sentence as a reminder of what we are working with
print(sentence1)
print(sentence2)

The supplier agreed to pay the manufacturer
The manufacturer agreed to pay the supplier


In [42]:
# Get embeddings for both sentences
emb1, tokens1 = get_bert_embedding(sentence1)
emb2, tokens2 = get_bert_embedding(sentence2)

In [43]:
# Review the output for supplier
tokens2.index('supplier')

7

In [44]:
# Compare the cosine similarity scores for supplier in each sentence
supplier_score = compare_word('supplier', emb1, tokens1, emb2, tokens2)

Comparing embeddings for word 'supplier':
Similarity score: 0.7416
Vector in sentence 1 (first 5 values): [ 0.9074208  -0.43041554  0.32430595  0.1308199   0.04717547]
Vector in sentence 2 (first 5 values): [ 0.19103658 -0.46897593  0.00414988  0.13056742 -0.34808037]


In [45]:
# Compare the cosine similarity scores for manufacturer in each sentence
manufacturer_score = compare_word('manufacturer', emb1, tokens1, emb2, tokens2)

Comparing embeddings for word 'manufacturer':
Similarity score: 0.8168
Vector in sentence 1 (first 5 values): [ 0.01716891 -0.26788452 -0.25919017  0.19550176 -0.46126062]
Vector in sentence 2 (first 5 values): [ 0.715219   -0.4425243   0.2655708   0.18022406 -0.00217637]


Same word but different embeddings! That's because the order in which they were used (sequence) in each sentence was used in the embedding.

## Try it!
Checkout the embeddings for another word of your choice.

In [46]:
compare_word('pay', emb1, tokens1, emb2, tokens2)

Comparing embeddings for word 'pay':
Similarity score: 0.9909
Vector in sentence 1 (first 5 values): [ 0.17565209 -0.01366376  0.34596705  0.2974591   0.11193716]
Vector in sentence 2 (first 5 values): [ 0.17775793 -0.12184241  0.31690663  0.31182256  0.03333706]


np.float32(0.99090314)

Notice that "pay" has a very high similarity across the two sentences.

## Conclusion
In this notebook, we learned how to use:
1. Word embeddings to represent text data
2. Word2Vec model to generate static word embeddings for a list of reviews
   - Word2Vec is a static embedding that has the same numerical representation for a given word regardless of context
   - Its performance varies based on how it is tuned and the data used for training
3. Pre-trained BERT model to generate contextual embeddings for a pair of sentences.
   - BERT is a contextual embedding that creates different embeddings for the same word depending on how it was used in a sentence.

### Recommended readings
- [Foundational word2vec paper](https://arxiv.org/abs/1301.3781)
- [Foundational BERT paper](https://arxiv.org/pdf/1810.04805)